# 🎲 Comparación multi-semilla: Baseline vs Logit Adjustment (Mac / MPS)

Corre los dos métodos que empataron (cross-entropy ponderada y logit adjustment) con **varias semillas**, para reportar **media ± desviación** en vez de una sola corrida. Así el empate queda respaldado estadísticamente y se cuantifica la varianza propia de tener clases tan pequeñas.

**Diseño:** el split de test se fija (semilla 42), igual para todas las corridas, así la sensibilidad es comparable (mismos 11 casos de riesgo). Lo que varía entre corridas es la **estocasticidad del entrenamiento**: inicialización del modelo, orden de los datos y augmentación.

⚠️ **Tiempo:** son 2 métodos × 3 semillas = **6 entrenamientos**. En MPS puede tomar 1,5–2,5 horas. Si tienes poco tiempo, baja `SEEDS` a 2 semillas (celda 1).

---
**Uso:** en `notebooks/`. Lee `../data/processed/dataset_clean.csv`. Restart + Run All.

## 1 · Configuración

In [ ]:
import numpy as np, pandas as pd, random
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix

DEVICE='cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
MODELO='PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'; MODELO_ALT='BSC-LT/roberta-base-biomedical-clinical-es'
MAX_LENGTH=256; LR=3e-5; EPOCHS=15; PATIENCE=4; BATCH=16

SEEDS=[42,123,7]                       # baja a [42,123] si quieres menos tiempo
METODOS=['baseline','logit_adjustment']
SPLIT_SEED=42                          # el split de test se fija para todas las corridas
print("Dispositivo:", DEVICE, "| semillas:", SEEDS)

## 2 · Datos y split FIJO (semilla 42)

In [ ]:
df=pd.read_csv('../data/processed/dataset_clean.csv', encoding='utf-8')
X=df['auditor_input'].values; y=df['BI-RADS'].values
Xtv,Xte,ytv,yte=train_test_split(X,y,test_size=0.15,random_state=SPLIT_SEED,stratify=y)
Xtr,Xva,ytr,yva=train_test_split(Xtv,ytv,test_size=0.176,random_state=SPLIT_SEED,stratify=ytv)
print(f"Train {len(Xtr)} | Val {len(Xva)} | Test {len(Xte)} (riesgo fijo: {(yte>=4).sum()})")

## 3 · Pérdidas, dataset y modelo

In [ ]:
class LogitAdjustLoss(nn.Module):
    def __init__(self, cls_num_list, tau=1.0):
        super().__init__()
        p=np.array(cls_num_list,dtype=np.float64); p=p/p.sum()
        self.adj=torch.tensor(np.log(p+1e-12)*tau,dtype=torch.float32)
    def forward(self,x,t): return F.cross_entropy(x+self.adj.to(x.device), t)

try: TOK=AutoTokenizer.from_pretrained(MODELO)
except Exception: MODELO=MODELO_ALT; TOK=AutoTokenizer.from_pretrained(MODELO)
class DS(Dataset):
    def __init__(self,t,l): self.t=list(t); self.l=list(l)
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=TOK(str(self.t[i]),truncation=True,padding='max_length',max_length=MAX_LENGTH,
              return_tensors='pt',return_token_type_ids=False)
        return {'input_ids':e['input_ids'].squeeze(0),'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.l[i],dtype=torch.long)}
class Modelo(nn.Module):
    def __init__(self,m,n=7,dp=0.5):
        super().__init__(); self.enc=AutoModel.from_pretrained(m)
        self.dp=nn.Dropout(dp); self.cl=nn.Linear(self.enc.config.hidden_size,n)
    def forward(self,ids,mask):
        return self.cl(self.dp(self.enc(input_ids=ids,attention_mask=mask).last_hidden_state[:,0,:]))
def aumentar(t):
    V=[('bordes irregulares','márgenes irregulares'),('bordes espiculados','márgenes espiculados'),
       ('bordes mal definidos','límites imprecisos'),('imagen nodular','nódulo'),('nódulo','imagen nodular'),
       ('lesión nodular','nódulo'),('heterogéneamente densas','de densidad heterogénea'),
       ('muy densas','extremadamente densas'),('calcificaciones sospechosas','depósitos cálcicos sospechosos'),
       ('microcalcificaciones','calcificaciones puntiformes agrupadas'),('mama derecha','hemimama derecha'),
       ('mama izquierda','hemimama izquierda'),('se visualiza','se observa'),('se observa','se visualiza'),
       ('se evidencia','se observa')]
    s=t
    for o,r in random.sample(V,min(3,len(V))):
        if o in s: s=s.replace(o,r,1)
    return s
print("Listo.")

## 4 · Función de entrenamiento (re-sembrada por corrida)

In [ ]:
def entrenar(metodo, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE=='mps': torch.mps.manual_seed(seed)
    # Augmentación (depende de la semilla)
    mask=np.isin(ytr,[3,4,5,6]); ta,la=[],[]
    for txt,lab in zip(Xtr[mask],ytr[mask]):
        for _ in range(3): ta.append(aumentar(txt)); la.append(lab)
    Xa=np.concatenate([Xtr,np.array(ta)]); ya=np.concatenate([ytr,np.array(la)])
    idx=np.random.RandomState(seed).permutation(len(Xa)); Xa,ya=Xa[idx],ya[idx]
    cls=np.array([(ya==c).sum() for c in range(7)],dtype=np.float64)

    tl=DataLoader(DS(Xa,ya),batch_size=BATCH,shuffle=True)
    vl=DataLoader(DS(Xva,yva),batch_size=BATCH); tt=DataLoader(DS(Xte,yte),batch_size=BATCH)
    model=Modelo(MODELO).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    sch=get_linear_schedule_with_warmup(opt,0,len(tl)*EPOCHS)
    pw=compute_class_weight('balanced',classes=np.unique(ya),y=ya); pv=np.ones(7,dtype=np.float32)
    for c,w in zip(np.unique(ya),pw): pv[c]=w
    crit = LogitAdjustLoss(cls,1.0) if metodo=='logit_adjustment' else nn.CrossEntropyLoss(weight=torch.tensor(pv).to(DEVICE))

    def ep(loader,train=True):
        model.train() if train else model.eval(); P=[]; R=[]
        with torch.set_grad_enabled(train):
            for b in loader:
                ids=b['input_ids'].to(DEVICE); mk=b['attention_mask'].to(DEVICE); lb=b['labels'].to(DEVICE)
                if train: opt.zero_grad()
                lo=model(ids,mk); loss=crit(lo,lb)
                if train: loss.backward(); opt.step(); sch.step()
                P.extend(lo.argmax(1).cpu().numpy()); R.extend(lb.cpu().numpy())
        return f1_score(R,P,average='macro',zero_division=0)
    mejor,sin=0,0; ruta=f'tmp_{metodo}_{seed}.pt'
    for e in range(1,EPOCHS+1):
        _=ep(tl,True); vf=ep(vl,False)
        if vf>mejor: mejor,sin=vf,0; torch.save(model.state_dict(),ruta)
        else: sin+=1
        if sin>=PATIENCE: break
    model.load_state_dict(torch.load(ruta,map_location=DEVICE)); model.eval()
    preds=[]; probs=[]; reales=[]
    with torch.no_grad():
        for b in tt:
            ids=b['input_ids'].to(DEVICE); mk=b['attention_mask'].to(DEVICE)
            pr=torch.softmax(model(ids,mk).float(),1).cpu().numpy()
            probs.extend(pr); preds.extend(pr.argmax(1)); reales.extend(b['labels'].numpy())
    preds=np.array(preds); reales=np.array(reales); probs=np.array(probs)
    f1m=f1_score(reales,preds,average='macro',zero_division=0)
    prisk=probs[:,4:7].sum(1); yz=(reales>=4).astype(int); auc=roc_auc_score(yz,prisk)
    yp=(prisk>=0.5).astype(int); tn,fp,fn,tp=confusion_matrix(yz,yp,labels=[0,1]).ravel()
    sens=tp/(tp+fn) if (tp+fn) else 0; espec=tn/(tn+fp) if (tn+fp) else 0
    return {'f1':f1m,'auc':auc,'sens':sens,'espec':espec}

## 5 · Ejecutar todas las corridas (6 entrenamientos)

In [ ]:
resultados={m:[] for m in METODOS}
for metodo in METODOS:
    for s in SEEDS:
        print(f"▶ {metodo} | semilla {s} ...", flush=True)
        r=entrenar(metodo,s); resultados[metodo].append(r)
        print(f"   F1={r['f1']:.4f} | AUC(d)={r['auc']:.3f} | Sens={r['sens']:.3f} | Espec={r['espec']:.3f}", flush=True)
print("\nTodas las corridas terminadas.")

## 6 · Resumen: media ± desviación

In [ ]:
def ms(vals): a=np.array(vals); return a.mean(), a.std()
print("="*64)
print("COMPARACIÓN MULTI-SEMILLA (test fijo, n_semillas={})".format(len(SEEDS)))
print("="*64)
print(f"{'Método':22} | {'F1-macro':>15} | {'Sens':>13} | {'Espec':>13}")
print("-"*64)
for m in METODOS:
    nom={'baseline':'CE ponderada','logit_adjustment':'Logit Adjustment'}[m]
    f1s=[r['f1'] for r in resultados[m]]; se=[r['sens'] for r in resultados[m]]; es=[r['espec'] for r in resultados[m]]
    print(f"{nom:22} | {np.mean(f1s):.4f} ± {np.std(f1s):.4f} | {np.mean(se):.3f} ± {np.std(se):.3f} | {np.mean(es):.3f} ± {np.std(es):.3f}")
print("="*64)
print("\nDetalle por semilla (F1-macro):")
for m in METODOS:
    print(f"  {m}: " + ", ".join(f"{r['f1']:.4f}" for r in resultados[m]))

## 7 · Cómo interpretar

- **Si los intervalos (media ± desviación) se solapan**, los dos métodos son estadísticamente equivalentes: el empate es real, no casualidad. Reportas cualquiera de los dos (o el baseline, por simplicidad).
- **La desviación misma es un resultado**: cuantifica cuánto fluctúa el F1 entre corridas por la pequeñez de las clases. Una desviación alta (p. ej. ±0,03 o más) respalda que las diferencias de una sola corrida no son confiables.
- Para la tesis: *"Promediando N semillas, la ponderación de clases y el logit adjustment fueron equivalentes (F1-macro X ± Y vs. X ± Y), confirmando que ningún método de balanceo supera la limitación de datos."*